# Assignment 2: Food Delivery Time Prediction (FINAL CORRECT)

Includes Regression + Classification + Feature Engineering + Visualizations

## Phase 1 - Data Preprocessing & Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler

df = pd.read_csv("Food_Delivery_Time_Prediction.csv")
df.head()

In [ ]:
df = df.fillna(df.mean(numeric_only=True))

In [ ]:
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

df['Distance'] = df.apply(lambda x: haversine(x['Restaurant_Latitude'], x['Restaurant_Longitude'],
                                              x['Delivery_Latitude'], x['Delivery_Longitude']), axis=1)

In [ ]:
scaler = StandardScaler()
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = scaler.fit_transform(df[num_cols])

In [ ]:
df['Fast_Delivery'] = (df['Delivery_Time'] < df['Delivery_Time'].median()).astype(int)

## Visualization

In [ ]:
sns.histplot(df['Delivery_Time'])
plt.title("Delivery Time Distribution")
plt.show()

In [ ]:
sns.heatmap(df.corr(), cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

## Linear Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X = df.drop(['Delivery_Time','Fast_Delivery'], axis=1)
y = df['Delivery_Time']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

## Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X = df.drop(['Delivery_Time','Fast_Delivery'], axis=1)
y = df['Fast_Delivery']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

## Insights
- Distance strongly affects delivery time.
- Traffic/weather increase delays.
- Model performs well for classification.